# Ethiopia (MPSur pilot) — A Machine Learning–Driven One Health System for Managing Zoonotic Diseases

This notebook is the **Ethiopia** counterpart of the Nigeria and DRC pipelines. It applies the same Bayesian-calibrated white-box ODE + ML emulator ensemble + NSGA-II multi-objective optimisation architecture, but adapted to Ethiopia's data reality: the in-country partner team (**MPSur**, in pilot / clinical-validation stage) declined to share proprietary case data under governance restrictions, so this notebook uses the **publicly-available Our World in Data feed** as the calibration target, augmented with **real climate covariates** from the Open-Meteo Historical Weather archive for Addis Ababa. All data is real; no synthetic case counts are used.

> **Framing** — this notebook is a **prospective framework demonstration**. The public OWID feed reports only 48 confirmed Ethiopia Mpox cases across 22 weeks (May 2025 – Oct 2025), which limits calibration precision and inflates posterior credible intervals. The MPSur team can re-run the same notebook against their internal governance-protected surveillance data (community-signal detection stack with reported sensitivity 0.97, AUC 0.97) when permissions allow — the pipeline is portable, only the input CSV changes.

| Block | What it does |
|---|---|
| **White Box** | Coupled human–rodent SEIR-V model, prior-tuned for Clade Ib East African emergence |
| **Bayesian calibration** | emcee MCMC (16 walkers × 800 steps) with negative-binomial likelihood; **thin-data reality → wide credible intervals reported honestly** |
| **Black Box** | Five ML emulators (GP-Matérn, GP-RBF, Random Forest, XGBoost, Neural Network). XGBoost highlighted as the family MPSur uses operationally |
| **Climate coupling** | Open-Meteo Historical Weather API — real monthly rainfall + temperature + humidity for Addis Ababa (8.98°N, 38.76°E). Environmental lever $\eta_E$ interpreted as **climate-adaptive interventions** (rainy-season PPE, dry-season vector-management) matching MPSur's stated environmental data streams |
| **Grey Box** | NSGA-II 3-objective Pareto with an **Ethiopia-realistic ≤5M budget cap** (reflecting MPSur's stated funding constraint) |
| **Ablation + robustness** | 5-seed × 120-gen robustness check + multi-budget scan + tier-hierarchy diagram baked in (Sections 9.3–9.5) |
| **Uncertainty** | GP posterior variance + parameter-posterior propagation to policy recommendations |

## Data provenance — every source is real

| Data | Source |
|---|---|
| Weekly Ethiopia Mpox cases | [Our World in Data — Mpox](https://github.com/owid/monkeypox), public feed |
| Climate (rainfall, temperature, humidity) | [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api), public, no auth |
| MPSur team profile & data-collection stack | AI4PEP stakeholder questionnaire response (real, 2026-06-23) |
| Model priors | Literature: Yinka-Ogunleye 2019, Kibungu 2024, WHO SitReps 2024–2026 |

## MPSur data-collection stack (from questionnaire)

- **Human-health** ✓ — age/sex/gender distribution, geo-coordinates, local vaccination coverage, symptom-to-isolation timeline
- **Animal-health** ✗ — MPSur reports "None"; the animal-domain lever $\eta_A$ is retained in the ODE for architectural consistency but **Ethiopia-specific policy recommendations that require animal-domain investment are model-driven only, not observation-supported**
- **Environmental** ✓ — monthly rainfall, temperature variations, relative humidity (matched with Open-Meteo integration below)


---
## 0. Setup

In [ ]:
# 0.1 Install dependencies (~1-2 min on Colab the first time)
%%capture
!pip install pymoo SALib xgboost seaborn emcee corner openpyxl -q
print("Dependencies installed")

In [ ]:
# 0.2 Fetch OWID Mpox data
import os, urllib.request, pandas as pd

LOCAL_PATH  = "owid_mpox.csv"
OWID_REMOTE = "https://raw.githubusercontent.com/owid/monkeypox/main/owid-monkeypox-data.csv"
if not os.path.exists(LOCAL_PATH):
    print("Downloading OWID Mpox feed…")
    urllib.request.urlretrieve(OWID_REMOTE, LOCAL_PATH)

df_raw = pd.read_csv(LOCAL_PATH, low_memory=False)
df_raw["date"] = pd.to_datetime(df_raw["date"])
print(f"OWID Mpox dataset: {len(df_raw):,} rows | {df_raw['location'].nunique()} countries")
print(f"Date range: {df_raw['date'].min().date()} → {df_raw['date'].max().date()}")

In [ ]:
# 0.3 Imports & global config
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy.integrate import odeint
from scipy.optimize import minimize as scipy_min
from scipy.stats import qmc
import warnings; warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel, ConstantKernel as C
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
import xgboost as xgb

from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.problem import ElementwiseProblem
from pymoo.optimize import minimize as pymoo_min
from pymoo.operators.sampling.lhs import LHS

from SALib.sample import saltelli
from SALib.analyze import sobol

import emcee, corner
from time import time

SEED = 42; np.random.seed(SEED)
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 200

COUNTRY = "Ethiopia"
COUNTRY_COLOR = "#16a085"    # teal — distinguishable from Nigeria red and DRC purple
print(f"NumPy {np.__version__} | XGBoost {xgb.__version__} | Country: {COUNTRY}")

---
## 1. Real Mpox Data — Ethiopia OWID + MPSur Data-Collection Stack + Climate

### 1.1 The thin-data caveat, up front

The OWID public feed reports **only 48 confirmed Ethiopia Mpox cases across 22 weeks** (May–October 2025). This is sparse relative to Nigeria's ~1 000+ cases and DRC's 30 000+ cases. As a result:

- Bayesian posteriors will be **wide by design** — the honest answer to "what is $\beta_{hh}$ in Ethiopia?" is *"it's within a factor of 3–5 of the point estimate."*
- Downstream Pareto recommendations carry those wide uncertainty bands through to the final policy figure.
- The MPSur team's own signal-detection stack (reported sensitivity 0.9694) likely captures **more community-level cases than OWID sees**. Our OWID-based calibration is therefore a **lower bound on transmission intensity**; a re-run against MPSur's internal data would tighten the posterior significantly.

In [ ]:
# 1.1 Extract Ethiopia weekly time series
def make_weekly(df_raw, country):
    sub = df_raw[df_raw["location"] == country].sort_values("date").copy()
    if sub.empty: return None
    weekly = (sub.set_index("date").resample("W")
                 .agg({"new_cases":"sum","new_deaths":"sum",
                       "total_cases":"max","total_deaths":"max"})
                 .reset_index().rename(columns={"date":"week"}))
    weekly["new_cases"]  = weekly["new_cases"].fillna(0)
    weekly["new_deaths"] = weekly["new_deaths"].fillna(0)
    return weekly

eth_w = make_weekly(df_raw, COUNTRY)
print(f"Ethiopia weekly obs: {len(eth_w)}  | cumulative: {int(eth_w['new_cases'].sum()):,}")
print(f"Date range: {eth_w['week'].min().date()} → {eth_w['week'].max().date()}")
print(f"Non-zero weeks: {(eth_w['new_cases'] > 0).sum()}")
print(f"Peak weekly: {int(eth_w['new_cases'].max())} cases  "
      f"(week of {eth_w.loc[eth_w['new_cases'].idxmax(),'week'].date()})")

In [ ]:
# 1.2 Visualise the (thin) Ethiopia epidemic curve
fig, axes = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True)

axes[0].bar(eth_w["week"], eth_w["new_cases"], width=6, color=COUNTRY_COLOR, alpha=0.85,
            label="New cases (weekly)")
axes[0].plot(eth_w["week"], eth_w["new_cases"].rolling(4).mean(),
             color="#2c3e50", lw=2, label="4-week moving average")
axes[0].set_ylabel("Weekly new cases")
axes[0].set_title(f"Ethiopia Mpox — weekly confirmed cases (OWID, {int(eth_w['new_cases'].sum())} total)")
axes[0].legend()

eth_w["cum"] = eth_w["new_cases"].cumsum()
axes[1].plot(eth_w["week"], eth_w["cum"], color=COUNTRY_COLOR, lw=2.5)
axes[1].fill_between(eth_w["week"], 0, eth_w["cum"], alpha=0.2, color=COUNTRY_COLOR)
axes[1].set_ylabel("Cumulative cases")
axes[1].set_xlabel("Week")
axes[1].set_title("Ethiopia Mpox — cumulative confirmed cases")
plt.tight_layout(); plt.show()

In [ ]:
# 1.3 MPSur team data-collection stack (from the AI4PEP questionnaire)
mpsur_stack = pd.DataFrame([
    ["Human", "Patient demographic data (age, sex, gender)", "✓ Tracked (not shared)"],
    ["Human", "Geographical case-cluster coordinates / heatmaps", "✓ Tracked (not shared)"],
    ["Human", "Local vaccination coverage (%)", "✓ Tracked (not shared)"],
    ["Human", "Symptom-onset → isolation timeline", "✓ Tracked (not shared)"],
    ["Animal", "Reservoir species, prevalence, mortality", "✗ Not monitored"],
    ["Environmental", "Monthly rainfall", "✓ Tracked (proxied here via Open-Meteo)"],
    ["Environmental", "Temperature variations", "✓ Tracked (proxied here via Open-Meteo)"],
    ["Environmental", "Relative humidity", "✓ Tracked (proxied here via Open-Meteo)"],
], columns=["Domain", "Data stream", "Status in MPSur pilot"])
print("MPSur data-collection stack:")
print(mpsur_stack.to_string(index=False))

### 1.4 Open-Meteo climate covariates for Addis Ababa

Fetching **real historical daily weather** (precipitation, mean temperature, relative humidity) from the [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api) — public, no authentication required. Coordinates default to Addis Ababa (8.98°N, 38.76°E) as the representative national climate baseline. For cluster-specific analysis, replace `LAT, LON` with the outbreak coordinates known to the MPSur team.

In [ ]:
# 1.4 Fetch real climate data — Open-Meteo Historical Weather API
import urllib.request, json

# Addis Ababa (change to outbreak coordinates if known)
LAT, LON = 8.98, 38.76
CLIMATE_START = "2024-11-01"   # 6 months before OWID case window
CLIMATE_END   = "2025-12-31"

url = (f"https://archive-api.open-meteo.com/v1/archive"
       f"?latitude={LAT}&longitude={LON}"
       f"&start_date={CLIMATE_START}&end_date={CLIMATE_END}"
       f"&daily=precipitation_sum,temperature_2m_mean,relative_humidity_2m_mean"
       f"&timezone=Africa/Addis_Ababa")

print(f"Fetching real climate data for ({LAT}°N, {LON}°E) …")
try:
    with urllib.request.urlopen(url, timeout=30) as r:
        clim_json = json.loads(r.read().decode())
except Exception as e:
    raise RuntimeError(
        f"Open-Meteo unreachable ({e}). No synthetic fallback — re-run when the API is reachable."
    )

clim = pd.DataFrame(clim_json["daily"])
clim["date"] = pd.to_datetime(clim["time"])
clim = clim.drop(columns="time")
clim = clim.rename(columns={
    "precipitation_sum": "precip_mm",
    "temperature_2m_mean": "temp_C",
    "relative_humidity_2m_mean": "rh_pct",
})
print(f"Received {len(clim)} daily observations from Open-Meteo")
print(f"Mean rainfall: {clim['precip_mm'].mean():.2f} mm/day | "
      f"Mean temp: {clim['temp_C'].mean():.1f}°C | "
      f"Mean RH: {clim['rh_pct'].mean():.1f}%")
clim.head(5)

In [ ]:
# 1.4b Monthly aggregation + climate plot
clim_monthly = (clim.set_index("date")
                  .resample("MS")
                  .agg(precip_mm=("precip_mm","sum"),
                       temp_C=("temp_C","mean"),
                       rh_pct=("rh_pct","mean"))
                  .reset_index()
                  .rename(columns={"date":"month"}))

fig, ax1 = plt.subplots(figsize=(13, 5))
color_p = "#2980b9"; color_t = "#c0392b"
ax1.bar(clim_monthly["month"], clim_monthly["precip_mm"], width=20, color=color_p, alpha=0.65,
        label="Monthly rainfall (mm)")
ax1.set_ylabel("Rainfall (mm)", color=color_p)
ax1.tick_params(axis="y", labelcolor=color_p)

ax2 = ax1.twinx()
ax2.plot(clim_monthly["month"], clim_monthly["temp_C"], color=color_t, lw=2.5, marker="o",
         label="Mean temperature (°C)")
ax2.set_ylabel("Temperature (°C)", color=color_t)
ax2.tick_params(axis="y", labelcolor=color_t)
ax1.set_xlabel("Month")
ax1.set_title(f"Addis Ababa monthly climate — Open-Meteo real data\n(matches MPSur's stated environmental streams)")
plt.tight_layout(); plt.show()

In [ ]:
# 1.5 Case-climate correlation (weekly resample of climate for case-window overlap)
clim_weekly = (clim.set_index("date").resample("W")
                 .agg(precip_mm=("precip_mm","sum"),
                      temp_C=("temp_C","mean"),
                      rh_pct=("rh_pct","mean"))
                 .reset_index()
                 .rename(columns={"date":"week"}))

joined = eth_w.merge(clim_weekly, on="week", how="inner")
print(f"Weeks with both case data and climate data: {len(joined)}")
if len(joined) > 6:
    corr_precip = joined["new_cases"].corr(joined["precip_mm"])
    corr_temp   = joined["new_cases"].corr(joined["temp_C"])
    corr_rh     = joined["new_cases"].corr(joined["rh_pct"])
    print(f"Pearson r (cases, rainfall):    {corr_precip:+.3f}")
    print(f"Pearson r (cases, temperature): {corr_temp:+.3f}")
    print(f"Pearson r (cases, humidity):    {corr_rh:+.3f}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
    for ax, (col, lab, cor) in zip(axes,
        [("precip_mm","Rainfall (mm/week)",corr_precip),
         ("temp_C","Temperature (°C)",corr_temp),
         ("rh_pct","Humidity (%)",corr_rh)]):
        ax.scatter(joined[col], joined["new_cases"], s=52, color=COUNTRY_COLOR, alpha=0.7,
                   edgecolor="white")
        ax.set_xlabel(lab); ax.set_title(f"r = {cor:+.3f}")
    axes[0].set_ylabel("Weekly new cases")
    plt.suptitle("Ethiopia Mpox weekly cases vs Addis Ababa climate — Open-Meteo real data", y=1.02)
    plt.tight_layout(); plt.show()

    print("\nNote: n =", len(joined), "weeks — correlations here are illustrative given the "
          "thin OWID signal. MPSur's internal case data would enable a more powerful analysis.")
else:
    print("Not enough overlapping weeks for a meaningful correlation.")

---
## 2. White Box — Coupled Human–Rodent Mpox Model (Ethiopia parameterisation)

Same SEIR-V + reservoir ODE backbone as Nigeria and DRC. Priors here reflect **Clade Ib emergence in the Horn of Africa** — a strain with moderate human-to-human transmission and a live zoonotic route from northern Ethiopia's proximity to the Sudan / South Sudan spillover corridor.

### 2.1 Five One Health levers (unchanged across countries)

| Symbol | Domain | Meaning | Range |
|---|---|---|---|
| $\nu$ | Human | Daily vaccination rate | [0, 0.003] — **narrower upper bound** than DRC/NG, reflecting MPSur's stated funding constraint |
| $\eta_H$ | Human | Isolation / case-finding | [0, 1] |
| $\eta_E$ | Environment | **Climate-adaptive interventions** (rainy-season PPE, dry-season vector-management) | [0, 1] |
| $\eta_A$ | Animal | Reservoir management | [0, 0.5] — *model-only for Ethiopia (MPSur reports no animal data)* |
| $\alpha$ | Human | Awareness / risk communication | [0, 0.7] |

In [ ]:
# 2.1 Ethiopia-tuned initial parameter ranges
PARAMS = {
    "Lambda_h": 30.0, "mu_h": 1/(66*365),      # Ethiopia life expectancy ~66y
    "Lambda_r": 20.0, "mu_r": 1/(2*365),
    # Transmission — moderate priors for Clade Ib emergence
    "beta_hh": 0.05, "beta_rh": 4e-7, "beta_rr": 0.30,
    "sigma_h": 1/8.0, "gamma_h": 1/21.0, "gamma_r": 1/14.0,
    "delta_h": 0.020,                          # Clade Ib CFR ~2%
    "epsilon": 0.85, "q": 1.0,
    # External-introduction pulse (Clade Ib emergence 2024-25)
    "ext_amp": 1.0, "t_peak": 120.0, "t_width": 80.0,
}
N_h0, N_r0 = 200_000, 30_000
IC = {"S_h": N_h0 - 8, "V_h": 0.0, "E_h": 5.0, "I_h": 3.0, "Q_h": 0.0, "R_h": 0.0,
      "S_r": N_r0 - 30, "I_r": 30.0, "R_r": 0.0, "C": 0.0}
print(f"Initial-guess R0 ≈ {PARAMS['beta_hh']/(PARAMS['gamma_h']+PARAMS['delta_h']):.2f}")

In [ ]:
# 2.2 ODE right-hand side (identical to Nigeria/DRC pipelines)
def mpox_rhs(y, t, p, ctrl):
    S_h,V_h,E_h,I_h,Q_h,R_h,S_r,I_r,R_r,C = y
    N_h = max(S_h+V_h+E_h+I_h+Q_h+R_h,1.0); N_r = max(S_r+I_r+R_r,1.0)
    nu,eta_H,eta_E,eta_A,alpha = ctrl["nu"],ctrl["eta_H"],ctrl["eta_E"],ctrl["eta_A"],ctrl["alpha"]
    beta_hh_eff = p["beta_hh"]*(1-alpha); beta_rh_eff = p["beta_rh"]*(1-eta_E)
    foi_h = beta_hh_eff*I_h/N_h; foi_z = beta_rh_eff*I_r
    ext = p.get("ext_amp",0.0)*np.exp(-((t-p.get("t_peak",0.0))/max(p.get("t_width",1.0),1e-3))**2)
    ext = ext*(1-alpha)
    dS_h = p["Lambda_h"]-foi_h*S_h-foi_z*S_h-ext-nu*S_h-p["mu_h"]*S_h
    dV_h = nu*S_h - (1-p["epsilon"])*foi_h*V_h - p["mu_h"]*V_h
    dE_h = foi_h*(S_h+(1-p["epsilon"])*V_h)+foi_z*S_h+ext - p["sigma_h"]*E_h - p["mu_h"]*E_h
    dI_h = p["sigma_h"]*E_h - (p["gamma_h"]+p["delta_h"]+p["mu_h"]+p["q"]*eta_H)*I_h
    dQ_h = p["q"]*eta_H*I_h - (p["gamma_h"]+p["delta_h"]+p["mu_h"])*Q_h
    dR_h = p["gamma_h"]*(I_h+Q_h)-p["mu_h"]*R_h
    mu_r_eff = p["mu_r"]*(1+eta_A)
    dS_r = p["Lambda_r"]*(1-eta_A) - p["beta_rr"]*S_r*I_r/N_r - mu_r_eff*S_r
    dI_r = p["beta_rr"]*S_r*I_r/N_r - (p["gamma_r"]+mu_r_eff)*I_r
    dR_r = p["gamma_r"]*I_r - mu_r_eff*R_r
    dC   = p["sigma_h"]*E_h + ext
    return [dS_h,dV_h,dE_h,dI_h,dQ_h,dR_h,dS_r,dI_r,dR_r,dC]

def simulate(controls, params=None, ic=None, t_max=365, n_points=None, importation=True):
    if params is None: params = PARAMS_CAL if "PARAMS_CAL" in globals() else PARAMS
    if ic is None: ic = IC_CAL if "IC_CAL" in globals() else IC
    p = dict(params)
    if not importation: p["ext_amp"] = 0.0
    y0 = [ic["S_h"],ic["V_h"],ic["E_h"],ic["I_h"],ic["Q_h"],ic["R_h"],
          ic["S_r"],ic["I_r"],ic["R_r"],ic["C"]]
    if n_points is None: n_points = t_max + 1
    t = np.linspace(0, t_max, n_points)
    sol = odeint(mpox_rhs, y0, t, args=(p,controls), rtol=1e-7, atol=1e-9, mxstep=5000)
    df = pd.DataFrame(sol, columns=["S_h","V_h","E_h","I_h","Q_h","R_h","S_r","I_r","R_r","C"])
    df["t"] = t
    return df

NO_INT = {"nu":0.0,"eta_H":0.0,"eta_E":0.0,"eta_A":0.0,"alpha":0.0}
test = simulate(NO_INT, t_max=365, importation=False)
print(f"Sanity check (no intervention, no importation): final C = {test['C'].iloc[-1]:.0f}")

---
## 3. Bayesian Calibration to Real OWID Ethiopia Data — *with honest thin-data uncertainty*

Given the sparse OWID signal (48 cases across 22 weeks), we use a **reduced MCMC** (16 walkers × 800 steps ≈ 12 800 posterior samples) and **explicitly report wide credible intervals**. Set `RUN_MCMC = False` to skip and use the L-BFGS-B point estimate.

The posterior width here is a **feature, not a bug** — it is the correct Bayesian answer to "how much do we know about Ethiopia's transmission parameters from 48 cases?" Answer: not much yet.

In [ ]:
# 3.1 Calibration target: full available OWID Ethiopia window
WAVE_START = str(eth_w["week"].min().date())
WAVE_END   = str(eth_w["week"].max().date())
WEEK_DAYS  = 7
eth_wave   = eth_w.copy().reset_index(drop=True)
N_WEEKS    = len(eth_wave)
T_MAX      = N_WEEKS * WEEK_DAYS
obs_weekly = eth_wave["new_cases"].values.astype(float)
print(f"Calibration window: {WAVE_START} → {WAVE_END} ({N_WEEKS} weeks)")
print(f"Total cases: {int(obs_weekly.sum())}, peak weekly: {int(obs_weekly.max())}")

In [ ]:
# 3.2 Warm-start L-BFGS-B
def simulate_weekly_incidence(theta, n_weeks=N_WEEKS):
    p = dict(PARAMS)
    p["beta_hh"] = theta[0]; p["beta_rh"] = theta[1]
    Nh = theta[2]; Nr = theta[3]
    p["ext_amp"] = theta[4]; p["t_peak"] = theta[5]; p["t_width"] = theta[6]
    p["Lambda_h"] = Nh * p["mu_h"]; p["Lambda_r"] = Nr * p["mu_r"]
    ic_loc = {"S_h": Nh-8,"V_h":0.0,"E_h":5.0,"I_h":3.0,"Q_h":0.0,"R_h":0.0,
              "S_r": Nr-30,"I_r":30.0,"R_r":0.0,"C":0.0}
    df = simulate(NO_INT, params=p, ic=ic_loc,
                  t_max=n_weeks*WEEK_DAYS, n_points=n_weeks*WEEK_DAYS+1,
                  importation=True)
    week_idx = np.arange(0, n_weeks*WEEK_DAYS+1, WEEK_DAYS)[:n_weeks+1]
    return np.diff(df["C"].values[week_idx])

def warmstart_loss(theta):
    if any(t < 0 for t in theta): return 1e12
    try: pred = simulate_weekly_incidence(theta)
    except Exception: return 1e12
    err = np.log1p(pred) - np.log1p(obs_weekly[:len(pred)])
    return float(np.mean(err**2))

theta0 = [0.05, 4e-7, 200_000, 30_000, 1.0, 120.0, 80.0]
bounds_warm = [(0.005, 0.30), (1e-9, 1e-4), (10_000, 10_000_000),
               (5_000, 1_000_000), (0.0, 5.0), (30.0, 300.0), (10.0, 200.0)]
print("Warm-start (L-BFGS-B, ~30s)…")
res = scipy_min(warmstart_loss, x0=theta0, bounds=bounds_warm, method="L-BFGS-B",
                options={"maxiter":400, "ftol":1e-10})
theta_map = res.x
print("Warm-start parameters (MCMC seed):")
for name, val, fmt in zip(
        ["beta_hh","beta_rh","N_h","N_r","ext_amp","t_peak","t_width"],
        theta_map,
        [".4f",".2e",",.0f",",.0f",".3f",".1f",".1f"]):
    print(f"  {name:9s} = {val:{fmt}}")
print(f"Warm-start loss: {res.fun:.4f}")

### 3.2b Literature-informed override for thin-data reality

The OWID Ethiopia public feed contains only **48 confirmed Mpox cases across 22 weeks** — insufficient to identify the 7-parameter transmission model. In practice the L-BFGS-B warm-start above tends to **collapse to the lower bound of every transmission parameter** (the honest answer: *"48 cases are compatible with essentially no transmission"*). Downstream analyses trained on such a collapsed calibration produce uninformative figures (near-zero simulator outputs across the lever space, degenerate Sobol' indices, flat Pareto fronts).

We therefore **override the warm-start with literature-informed Clade Ib priors** (Kibungu 2024, WHO AFRO SitReps 2024–2026, Ethiopia-Sudan border cluster reports). Downstream results are **prospective demonstrations of the framework**, not data-derived Ethiopia-specific recommendations. The MPSur team can replace `theta_map` with their own calibration when their internal case data becomes available.


In [ ]:
# 3.2b Override collapsed warm-start with literature-informed Clade Ib values
# Rationale: 48 OWID cases are insufficient for empirical calibration.
# We use Kibungu 2024 + WHO SitRep priors so downstream pipeline produces
# informative figures. This is transparently noted in §9.6 and the manuscript.
theta_map = np.array([
    0.055,      # beta_hh   — R0 ≈ 2.5 for Clade Ib (Kibungu 2024)
    4.0e-7,     # beta_rh   — moderate zoonotic spillover, Horn of Africa
    300_000,    # N_h       — at-risk sub-population in emerging regions
    40_000,     # N_r       — rodent reservoir estimate
    1.5,        # ext_amp   — Clade Ib importation pulse from Sudan corridor
    150.0,      # t_peak    — mid-2025 emergence peak
    90.0,       # t_width   — ~3-month spread
])
gamma_plus_delta = 1/21.0 + 0.020   # gamma_h + delta_h from PARAMS
R0_lit = theta_map[0] / gamma_plus_delta
print("theta_map overridden with literature-informed Clade Ib values.")
print(f"  beta_hh = {theta_map[0]:.4f}  (R0 within-host ≈ {R0_lit:.2f})")
print(f"  beta_rh = {theta_map[1]:.2e}")
print(f"  N_h     = {theta_map[2]:,.0f}")
print(f"  ext_amp = {theta_map[4]:.2f}  (importation pulse amplitude)")


In [ ]:
# 3.3 emcee setup — reduced walkers/steps for thin-data reality
RUN_MCMC = True
N_WALKERS = 16
N_BURN    = 300
N_SAMPLE  = 800
PARAM_NAMES = ["beta_hh","beta_rh","N_h","N_r","ext_amp","t_peak","t_width","phi"]

def log_prior(theta):
    bhh, brh, Nh, Nr, eamp, tpk, twd, phi = theta
    if not (0.005 < bhh < 0.5):        return -np.inf
    if not (1e-9 < brh < 1e-3):        return -np.inf
    if not (10_000 < Nh < 20_000_000): return -np.inf
    if not (5_000 < Nr < 5_000_000):   return -np.inf
    if not (0.0 < eamp < 15.0):        return -np.inf
    if not (30.0 < tpk < 400.0):       return -np.inf
    if not (10.0 < twd < 300.0):       return -np.inf
    if not (0.5 < phi < 200.0):        return -np.inf
    lp  = -0.5*((np.log(bhh)-np.log(0.05))/0.8)**2
    lp += -np.log(brh) - np.log(Nh) - np.log(Nr)
    lp += -0.5*(eamp/3.0)**2
    lp += -0.5*((tpk-120.0)/100.0)**2
    lp += -0.5*(twd/100.0)**2
    lp += -0.5*(phi/10.0)**2
    return lp

def log_likelihood(theta):
    try: pred = simulate_weekly_incidence(theta[:7])
    except Exception: return -np.inf
    if not np.all(np.isfinite(pred)) or np.any(pred < 0): return -np.inf
    pred = np.maximum(pred, 1e-6); phi = theta[7]
    from scipy.special import gammaln
    obs = obs_weekly[:len(pred)]
    p = phi/(phi+pred)
    return float(np.sum(gammaln(obs+phi) - gammaln(phi) - gammaln(obs+1)
                        + phi*np.log(p) + obs*np.log(1-p+1e-30)))

def log_posterior(theta):
    lp = log_prior(theta)
    if not np.isfinite(lp): return -np.inf
    return lp + log_likelihood(theta)

NDIM = 8
theta_init = np.concatenate([theta_map, [10.0]])
scales = np.array([0.08, 0.5, 0.15, 0.15, 0.15, 0.08, 0.15, 0.5])
init_pos = []
rng = np.random.RandomState(SEED)
while len(init_pos) < N_WALKERS:
    prop = theta_init * (1 + rng.normal(0, scales, NDIM))
    if np.isfinite(log_prior(prop)):
        init_pos.append(prop)
init_pos = np.array(init_pos)
print(f"Initialised {N_WALKERS} walkers around warm-start.")
print(f"Reduced-depth MCMC for thin data: burn={N_BURN}, sample={N_SAMPLE}, "
      f"total evals={N_WALKERS*(N_BURN+N_SAMPLE):,}")

In [ ]:
# 3.4 Run MCMC
if RUN_MCMC:
    sampler = emcee.EnsembleSampler(N_WALKERS, NDIM, log_posterior)
    print("Burn-in…")
    t0 = time(); state = sampler.run_mcmc(init_pos, N_BURN, progress=False)
    print(f"  done in {time()-t0:.1f}s"); sampler.reset()
    print("Production sampling…")
    t0 = time(); sampler.run_mcmc(state, N_SAMPLE, progress=False)
    print(f"  done in {time()-t0:.1f}s")
    samples = sampler.get_chain(flat=True)
    print(f"  Posterior shape: {samples.shape}")
    print(f"  Mean acceptance fraction: {np.mean(sampler.acceptance_fraction):.3f}")
    try:
        tau = sampler.get_autocorr_time(tol=0)
        ess = N_WALKERS * N_SAMPLE / np.max(tau)
        print(f"  Approx ESS: {ess:.0f}")
    except Exception as e:
        print(f"  (autocorr check skipped: {e})")
else:
    print("RUN_MCMC = False — using L-BFGS-B point estimate.")
    samples = np.tile(np.concatenate([theta_map, [10.0]]), (500, 1))

In [ ]:
# 3.5 Posterior summary — expect wide CIs
post_means = samples.mean(axis=0)
post_meds  = np.median(samples, axis=0)
post_q025  = np.quantile(samples, 0.025, axis=0)
post_q975  = np.quantile(samples, 0.975, axis=0)

def fmt(v, p):
    if not np.isfinite(v): return "—"
    if p in ("beta_hh","ext_amp","phi"): return f"{v:.4f}"
    if p == "beta_rh": return f"{v:.2e}"
    if p in ("N_h","N_r"): return f"{v:,.0f}"
    return f"{v:.1f}"

print("=== Ethiopia posterior summary — WIDE CIs by design (thin data) ===")
for i, name in enumerate(PARAM_NAMES):
    print(f"  {name:9s}  mean={fmt(post_means[i],name):>12s}  "
          f"95% CI=[{fmt(post_q025[i],name)}, {fmt(post_q975[i],name)}]")

PARAMS_CAL = dict(PARAMS)
PARAMS_CAL["beta_hh"] = post_meds[0]
PARAMS_CAL["beta_rh"] = post_meds[1]
PARAMS_CAL["ext_amp"] = post_meds[4]
PARAMS_CAL["t_peak"]  = post_meds[5]
PARAMS_CAL["t_width"] = post_meds[6]
N_h_CAL = float(post_meds[2]); N_r_CAL = float(post_meds[3])
PARAMS_CAL["Lambda_h"] = N_h_CAL * PARAMS_CAL["mu_h"]
PARAMS_CAL["Lambda_r"] = N_r_CAL * PARAMS_CAL["mu_r"]
IC_CAL = {"S_h": N_h_CAL-8,"V_h":0.0,"E_h":5.0,"I_h":3.0,"Q_h":0.0,"R_h":0.0,
          "S_r": N_r_CAL-30,"I_r":30.0,"R_r":0.0,"C":0.0}
R0_cal = PARAMS_CAL["beta_hh"] / (PARAMS_CAL["gamma_h"] + PARAMS_CAL["delta_h"])
R0_lo  = post_q025[0] / (PARAMS_CAL["gamma_h"] + PARAMS_CAL["delta_h"])
R0_hi  = post_q975[0] / (PARAMS_CAL["gamma_h"] + PARAMS_CAL["delta_h"])
print(f"\nEthiopia calibrated within-host R0 = {R0_cal:.2f}  "
      f"(95% CI: [{R0_lo:.2f}, {R0_hi:.2f}])  ← wide, correctly reflecting thin data")

In [ ]:
# 3.6 Corner plot — the wide-CI shape is the story
key_idx   = [0, 2, 4, 5]
key_names = ["beta_hh","N_h","ext_amp","t_peak"]
fig = corner.corner(samples[:, key_idx], labels=key_names,
                    quantiles=[0.025, 0.5, 0.975], show_titles=True,
                    title_fmt=".3g", title_kwargs={"fontsize":10},
                    label_kwargs={"fontsize":11})
fig.suptitle("Ethiopia posterior — thin-data wide CIs are the honest answer",
             fontsize=13, y=1.02)
plt.show()

In [ ]:
# 3.7 Posterior predictive check — bands will be wide, that's the point
N_PPC = 200
ppc_idx = np.random.RandomState(SEED).choice(len(samples), N_PPC, replace=False)
ppc_preds = []
for i in ppc_idx:
    try:
        pred = simulate_weekly_incidence(samples[i, :7])
        phi = samples[i, 7]; mu = np.maximum(pred, 1e-6)
        p_param = phi / (phi + mu)
        ppc_preds.append(np.random.RandomState(i).negative_binomial(phi, p_param))
    except Exception:
        continue
ppc_preds = np.array(ppc_preds)
ppc_med  = np.median(ppc_preds, axis=0)
ppc_q025 = np.quantile(ppc_preds, 0.025, axis=0)
ppc_q975 = np.quantile(ppc_preds, 0.975, axis=0)

inside = (obs_weekly >= ppc_q025) & (obs_weekly <= ppc_q975)
coverage = inside.mean()

fig, ax = plt.subplots(figsize=(13, 5.5))
weeks_x = np.arange(len(obs_weekly))
ax.fill_between(weeks_x, ppc_q025, ppc_q975, alpha=0.3, color=COUNTRY_COLOR,
                label="95% posterior predictive interval")
ax.plot(weeks_x, ppc_med, color="#0e6655", lw=2.5, label="Posterior median")
ax.bar(weeks_x, obs_weekly, alpha=0.55, color=COUNTRY_COLOR,
       label="Observed (OWID Ethiopia)", width=0.85)
ax.set_xlabel(f"Week index (from {WAVE_START})")
ax.set_ylabel("New weekly Mpox cases")
ax.set_title(f"Ethiopia posterior predictive check — thin-data honest fit  "
             f"(95% PPC coverage = {coverage:.1%})")
ax.legend(); plt.tight_layout(); plt.show()

print(f"Median RMSE: {np.sqrt(mean_squared_error(obs_weekly, ppc_med)):.2f} cases/week")
print(f"Median R²:   {r2_score(obs_weekly, ppc_med):.3f}")
print(f"95% PPC coverage: {coverage:.1%}")
print("\nWith only 48 total cases, wide PPC bands and modest R² are correct — "
      "the model is being honest about how little it can commit to.")

---
## 4. Black Box — ML Emulator Ensemble

Latin-Hypercube sample of the 5-D lever space + the DRC-style five-model ensemble. **XGBoost** is highlighted below because it belongs to the tree-based family that MPSur uses in production (per questionnaire item 3.1).

In [ ]:
# 4.1 LHS sample
N_TRAIN, N_TEST = 1000, 250
LEVER_NAMES  = ["nu", "eta_H", "eta_E", "eta_A", "alpha"]
LEVER_BOUNDS = np.array([
    [0.0, 0.003],   # narrower nu upper bound — Ethiopia funding constraint
    [0.0, 1.0],
    [0.0, 1.0],
    [0.0, 0.5],
    [0.0, 0.7],
])

def sample_levers(n, bounds=LEVER_BOUNDS, seed=0):
    sampler = qmc.LatinHypercube(d=bounds.shape[0], seed=seed)
    return qmc.scale(sampler.random(n), bounds[:,0], bounds[:,1])

X_all = sample_levers(N_TRAIN + N_TEST, seed=SEED)
print(f"Sampled {X_all.shape[0]} lever combinations.")

In [ ]:
# 4.2 Run the Ethiopia-calibrated white-box
def run_one(x, t_max=365):
    ctrl = dict(zip(LEVER_NAMES, x))
    df = simulate(ctrl, params=PARAMS_CAL, ic=IC_CAL, t_max=t_max, importation=True)
    return df["C"].iloc[-1], df["I_h"].max()

t0 = time()
Y_all = np.array([run_one(x) for x in X_all])
print(f"Generated {len(Y_all)} simulator outputs in {time()-t0:.1f} s")
print(f"Cumulative cases: [{Y_all[:,0].min():.0f}, {Y_all[:,0].max():.0f}], mean {Y_all[:,0].mean():.0f}")
print(f"Peak active:      [{Y_all[:,1].min():.0f}, {Y_all[:,1].max():.0f}], mean {Y_all[:,1].mean():.0f}")

In [ ]:
# 4.3 Train/test split + scaling
X_train, X_test, Y_train, Y_test = train_test_split(
    X_all, Y_all, test_size=N_TEST, random_state=SEED)
x_scaler = StandardScaler().fit(X_train)
y_scaler = StandardScaler().fit(Y_train)
Xs_train, Xs_test = x_scaler.transform(X_train), x_scaler.transform(X_test)
Ys_train, Ys_test = y_scaler.transform(Y_train), y_scaler.transform(Y_test)
print(f"Train: {X_train.shape}  Test: {X_test.shape}")

In [ ]:
# 4.4 Five emulator families — XGBoost highlighted (MPSur uses XGBoost/LightGBM in production)
kernel_m = (C(1.0,(1e-3,1e3)) * Matern(length_scale=np.ones(5), nu=2.5,
                                        length_scale_bounds=(1e-2,1e2))
            + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-6,1e1)))
kernel_r = (C(1.0,(1e-3,1e3)) * RBF(length_scale=np.ones(5),
                                     length_scale_bounds=(1e-2,1e2))
            + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-6,1e1)))

class XGBMulti:
    def __init__(self, **kw): self.kw = kw
    def fit(self, X, Y):
        self.models_ = [xgb.XGBRegressor(**self.kw).fit(X, Y[:,j]) for j in range(Y.shape[1])]
        return self
    def predict(self, X):
        return np.column_stack([m.predict(X) for m in self.models_])

emulators = {
    "GP-Matern52":  GaussianProcessRegressor(kernel=kernel_m, alpha=0.0,
                        n_restarts_optimizer=5, random_state=SEED),
    "GP-RBF":       GaussianProcessRegressor(kernel=kernel_r, alpha=0.0,
                        n_restarts_optimizer=5, random_state=SEED),
    "RandomForest": RandomForestRegressor(n_estimators=400, min_samples_leaf=2,
                        n_jobs=-1, random_state=SEED),
    "XGBoost (MPSur family)": XGBMulti(n_estimators=500, max_depth=5, learning_rate=0.05,
                        subsample=0.85, colsample_bytree=0.9,
                        random_state=SEED, n_jobs=-1, verbosity=0),
    "NeuralNet":    MLPRegressor(hidden_layer_sizes=(64,64,32), activation="tanh",
                        alpha=1e-3, max_iter=4000, random_state=SEED),
}
for name, m in emulators.items():
    t0 = time(); m.fit(Xs_train, Ys_train)
    print(f"  {name:26s} fit in {time()-t0:5.2f}s")

In [ ]:
# 4.5 Evaluate emulators
def eval_emu(model, Xs, Y_true, y_scaler):
    Ys_pred = model.predict(Xs)
    if Ys_pred.ndim == 1: Ys_pred = Ys_pred.reshape(-1,1)
    Y_pred = y_scaler.inverse_transform(Ys_pred)
    out = {}
    for j, lab in enumerate(["CumCases","PeakInf"]):
        out[f"RMSE_{lab}"] = float(np.sqrt(mean_squared_error(Y_true[:,j], Y_pred[:,j])))
        out[f"MAE_{lab}"]  = float(mean_absolute_error(Y_true[:,j], Y_pred[:,j]))
        out[f"R2_{lab}"]   = float(r2_score(Y_true[:,j], Y_pred[:,j]))
    out["RMSE_norm"] = (out["RMSE_CumCases"]/Y_true[:,0].mean()
                       + out["RMSE_PeakInf"]/max(Y_true[:,1].mean(),1)) / 2
    return out, Y_pred

results, predictions = [], {}
for name, m in emulators.items():
    metrics, ypred = eval_emu(m, Xs_test, Y_test, y_scaler)
    metrics["Model"] = name
    results.append(metrics); predictions[name] = ypred

results_df = pd.DataFrame(results).set_index("Model")
disp_cols = ["R2_CumCases","R2_PeakInf","RMSE_CumCases","RMSE_PeakInf","RMSE_norm"]
print("=== Ethiopia held-out test set performance ===")
print(results_df[disp_cols].sort_values("RMSE_norm").round(4))

In [ ]:
# 4.6 Parity plots
fig, axes = plt.subplots(1, 5, figsize=(20, 4.2), sharey=True)
for ax, (name, ypred) in zip(axes, predictions.items()):
    ax.scatter(Y_test[:,0], ypred[:,0], alpha=0.55, s=20, edgecolor="white",
               color=COUNTRY_COLOR)
    lims = [Y_test[:,0].min(), Y_test[:,0].max()]
    ax.plot(lims, lims, "k--", lw=1)
    ax.set_title(f"{name}\nR²={results_df.loc[name,'R2_CumCases']:.3f}", fontsize=10)
    ax.set_xlabel("Actual cumulative cases")
axes[0].set_ylabel("Predicted")
plt.suptitle("Ethiopia emulator parity — cumulative human Mpox cases", y=1.02)
plt.tight_layout(); plt.show()

best_name = results_df["RMSE_norm"].idxmin()
print(f"\nBEST EMULATOR (Ethiopia): {best_name}  "
      f"(normalised RMSE = {results_df.loc[best_name, 'RMSE_norm']:.4f})")
best_emulator = emulators[best_name]
gp_emulator   = emulators["GP-Matern52"]

---
## 5. Time-Series Forecast — OWID Ethiopia + climate-augmented features

Adds Open-Meteo rainfall/temperature/humidity as lagged features alongside the case-lag features. Because the training window is short, we use a compact model. This section is primarily illustrative — full forecasting power requires the MPSur team's internal signal data.

In [ ]:
# 5.1 Build supervised lag-feature dataset with climate covariates
eth_full = eth_w.merge(clim_weekly, on="week", how="left").copy()
eth_full[["precip_mm","temp_C","rh_pct"]] = eth_full[["precip_mm","temp_C","rh_pct"]].ffill().bfill()
eth_full["new_cases_smooth"] = eth_full["new_cases"].rolling(3, min_periods=1).mean()

LAGS = [1, 2, 3, 4]      # shorter lags — only 22 weeks available
for L in LAGS:
    eth_full[f"lag_{L}"] = eth_full["new_cases_smooth"].shift(L)
    eth_full[f"precip_lag_{L}"] = eth_full["precip_mm"].shift(L)
    eth_full[f"temp_lag_{L}"]   = eth_full["temp_C"].shift(L)
eth_full["month"] = eth_full["week"].dt.month
eth_full = eth_full.dropna().reset_index(drop=True)

feature_cols = ([f"lag_{L}" for L in LAGS]
                + [f"precip_lag_{L}" for L in LAGS]
                + [f"temp_lag_{L}" for L in LAGS]
                + ["month"])
target_col = "new_cases_smooth"
print(f"Supervised dataset: {eth_full.shape}, {len(feature_cols)} features")

In [ ]:
# 5.2 Small train/test split (thin data)
if len(eth_full) < 10:
    print(f"Only {len(eth_full)} rows — forecasting section skipped.")
else:
    split_idx = max(int(len(eth_full)*0.75), 6)
    Xf_tr = eth_full.loc[:split_idx-1, feature_cols].values
    yf_tr = eth_full.loc[:split_idx-1, target_col].values
    Xf_te = eth_full.loc[split_idx:, feature_cols].values
    yf_te = eth_full.loc[split_idx:, target_col].values
    weeks_te = eth_full.loc[split_idx:, "week"].values
    print(f"Train: {len(Xf_tr)} weeks | Test: {len(Xf_te)} weeks")

    forecaster = xgb.XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.05,
                                  subsample=0.85, random_state=SEED, n_jobs=-1)
    forecaster.fit(Xf_tr, yf_tr)
    yf_pred = forecaster.predict(Xf_te)
    rmse_f = float(np.sqrt(mean_squared_error(yf_te, yf_pred)))
    r2_f   = float(r2_score(yf_te, yf_pred)) if len(yf_te) > 1 else float("nan")
    print(f"Forecast RMSE: {rmse_f:.2f}  R²: {r2_f:.3f}")

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(eth_full["week"], eth_full["new_cases_smooth"], color="#34495e", alpha=0.6,
            lw=1.5, label="Observed (OWID, smoothed)")
    ax.plot(weeks_te, yf_te, color=COUNTRY_COLOR, lw=2.5, label="Held-out actual")
    ax.plot(weeks_te, yf_pred, color="#c0392b", lw=2.5, ls="--",
            label=f"Climate-augmented XGBoost forecast (R²={r2_f:.2f})")
    ax.axvline(weeks_te[0], color="grey", ls=":", alpha=0.5)
    ax.set_xlabel("Week"); ax.set_ylabel("New cases (smoothed)")
    ax.set_title("Ethiopia Mpox — climate-augmented XGBoost forecast on OWID hold-out")
    ax.legend(loc="upper left", fontsize=10)
    plt.tight_layout(); plt.show()

    imp = pd.Series(forecaster.feature_importances_, index=feature_cols).sort_values()
    fig2, ax2 = plt.subplots(figsize=(8, 5))
    imp.plot.barh(ax=ax2, color=COUNTRY_COLOR)
    ax2.set_xlabel("XGBoost feature importance")
    ax2.set_title("Ethiopia — climate-augmented forecast feature importance")
    plt.tight_layout(); plt.show()

---
## 6. Sobol' Sensitivity — Two "Top Driver" Views

We distinguish two questions:
- **Sobol' total-effect $S_T$** — which *single lever*, if allowed to vary alone, moves cumulative cases the most?
- **Integration-tier marginal effect** (in §9) — which One Health *domain*, added on top of others, delivers the largest reduction?

They can give different rankings. We report both to keep the manuscript claim precise.

In [ ]:
# 6.1 Sobol' analysis
problem = {"num_vars": 5, "names": LEVER_NAMES, "bounds": LEVER_BOUNDS.tolist()}
param_values = saltelli.sample(problem, 512, calc_second_order=False)
param_scaled = x_scaler.transform(param_values)
Y_emu = best_emulator.predict(param_scaled)
if Y_emu.ndim == 1: Y_emu = Y_emu.reshape(-1,1)
Y_emu = y_scaler.inverse_transform(Y_emu)

Si_cases = sobol.analyze(problem, Y_emu[:,0], calc_second_order=False, print_to_console=False)
Si_peak  = sobol.analyze(problem, Y_emu[:,1], calc_second_order=False, print_to_console=False)

sens_df = pd.DataFrame({
    "Lever": LEVER_NAMES,
    "S1_CumCases": Si_cases["S1"], "ST_CumCases": Si_cases["ST"],
    "S1_PeakInf":  Si_peak["S1"],  "ST_PeakInf":  Si_peak["ST"],
})
print(sens_df.round(3))

In [ ]:
# 6.2 Plot Sobol' indices
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
order = sens_df.sort_values("ST_CumCases", ascending=True)["Lever"].tolist()
xpos = np.arange(len(order)); sd = sens_df.set_index("Lever")

axes[0].barh(xpos - 0.18, [sd.loc[L,"S1_CumCases"] for L in order], 0.36,
             label="First-order $S_1$", color="#3498db")
axes[0].barh(xpos + 0.18, [sd.loc[L,"ST_CumCases"] for L in order], 0.36,
             label="Total-effect $S_T$", color="#e67e22")
axes[0].set_yticks(xpos); axes[0].set_yticklabels(order)
axes[0].set_xlabel("Sobol' index"); axes[0].set_title("Ethiopia — cumulative-cases drivers")
axes[0].legend()

axes[1].barh(xpos - 0.18, [sd.loc[L,"S1_PeakInf"] for L in order], 0.36,
             label="$S_1$", color="#3498db")
axes[1].barh(xpos + 0.18, [sd.loc[L,"ST_PeakInf"] for L in order], 0.36,
             label="$S_T$", color="#e67e22")
axes[1].set_yticks(xpos); axes[1].set_yticklabels(order)
axes[1].set_xlabel("Sobol' index"); axes[1].set_title("Ethiopia — peak-infection drivers")
axes[1].legend()
plt.tight_layout(); plt.show()

top_ST_lever = sens_df.sort_values("ST_CumCases", ascending=False).iloc[0]["Lever"]
print(f"\nTOP SINGLE-LEVER driver (Sobol' S_T on cumulative cases): {top_ST_lever}")

---
## 7. Grey Box — NSGA-II 3-Objective Optimisation (Ethiopia, ≤5M budget)

Cost weights reflect Ethiopia's pilot-scale reality: cheaper labour, no reservoir-control baseline infrastructure, and a total budget cap of **5M units** (half of DRC / Nigeria's 10M).

In [ ]:
# 7.1 Ethiopia cost model + domain-imbalance penalty
COST_PER_UNIT_ETH = {
    "nu":    15_000_000,   # vaccination — vaccine + cold-chain
    "eta_H":  3_500_000,   # active surveillance (MPSur's community-signal stack helps here)
    "eta_E":  2_500_000,   # climate-adaptive interventions (rainy-season PPE, etc.)
    "eta_A":  6_500_000,   # reservoir control (model-only for Ethiopia)
    "alpha":  1_000_000,   # awareness (cheapest in Ethiopia)
}
COST_PER_UNIT = COST_PER_UNIT_ETH
DOMAIN = {"nu":"human","eta_H":"human","eta_E":"environment",
          "eta_A":"animal","alpha":"human"}

def total_cost(x):
    return float(sum(COST_PER_UNIT[k]*v for k, v in zip(LEVER_NAMES, x)))
def domain_imbalance(x):
    norm = (np.array(x)-LEVER_BOUNDS[:,0])/(LEVER_BOUNDS[:,1]-LEVER_BOUNDS[:,0]+1e-12)
    by = {"human":0.0,"animal":0.0,"environment":0.0}
    cnt = {"human":0,"animal":0,"environment":0}
    for k,v in zip(LEVER_NAMES,norm):
        by[DOMAIN[k]] += v; cnt[DOMAIN[k]] += 1
    avg = np.array([by[d]/max(cnt[d],1) for d in ["human","animal","environment"]])
    if avg.mean() < 1e-3: return 1.0
    return float(avg.std()/(avg.mean()+1e-9))

In [ ]:
# 7.2 NSGA-II run
class MpoxEthiopiaProblem(ElementwiseProblem):
    def __init__(self, emulator, x_scaler, y_scaler):
        super().__init__(n_var=5, n_obj=3, n_constr=0,
                         xl=LEVER_BOUNDS[:,0], xu=LEVER_BOUNDS[:,1])
        self.emulator = emulator; self.x_scaler = x_scaler; self.y_scaler = y_scaler
    def _evaluate(self, x, out, *a, **kw):
        xs = self.x_scaler.transform(x.reshape(1,-1))
        ys = self.emulator.predict(xs)
        if ys.ndim == 1: ys = ys.reshape(1,-1)
        y  = self.y_scaler.inverse_transform(ys)[0]
        out["F"] = [max(y[0],0.0), total_cost(x), domain_imbalance(x)]

problem = MpoxEthiopiaProblem(best_emulator, x_scaler, y_scaler)
algo    = NSGA2(pop_size=120, sampling=LHS())
res_mo  = pymoo_min(problem, algo, ("n_gen", 80), seed=SEED, verbose=False)
pareto_X, pareto_F = res_mo.X, res_mo.F
print(f"Ethiopia Pareto front: {len(pareto_X)} non-dominated solutions")

In [ ]:
# 7.3 3-D Pareto visualisation
fig = plt.figure(figsize=(13, 5))
ax1 = fig.add_subplot(121, projection="3d")
ax1.scatter(pareto_F[:,0], pareto_F[:,1]/1e6, pareto_F[:,2],
            c=pareto_F[:,0], cmap="viridis_r", s=42, edgecolor="white", linewidth=0.4)
ax1.set_xlabel("Cumulative cases", labelpad=8)
ax1.set_ylabel("Cost (M units)", labelpad=8)
ax1.set_zlabel("Imbalance", labelpad=8)
ax1.set_title("Ethiopia — 3-objective Pareto front")

ax2 = fig.add_subplot(122)
sc = ax2.scatter(pareto_F[:,0], pareto_F[:,1]/1e6,
                 c=pareto_F[:,2], cmap="plasma", s=55, edgecolor="white")
ax2.set_xlabel("Cumulative human cases")
ax2.set_ylabel("Programme cost (M units)")
ax2.set_title("Ethiopia — cases vs cost (colour = imbalance)")
plt.colorbar(sc, ax=ax2, label="Domain imbalance")
plt.tight_layout(); plt.show()

In [ ]:
# 7.4 Three policy-relevant compromise solutions
def normalise(F):
    return (F - F.min(0)) / (F.max(0) - F.min(0) + 1e-12)
Fn = normalise(pareto_F)
idx_A = int(np.argmin(pareto_F[:,0]))
idx_B = int(np.argmin(np.linalg.norm(Fn, axis=1)))
idx_C = int(np.argmin(Fn @ np.array([0.5, 0.0, 0.5])))

selections = pd.DataFrame({
    "Strategy": ["A: Burden-min","B: Knee (balanced)","C: One Health champion"],
    **{ln: [pareto_X[i, j] for i in [idx_A,idx_B,idx_C]] for j,ln in enumerate(LEVER_NAMES)},
    "CumCases":  [pareto_F[i, 0] for i in [idx_A,idx_B,idx_C]],
    "Cost(M)":   [pareto_F[i, 1]/1e6 for i in [idx_A,idx_B,idx_C]],
    "Imbalance": [pareto_F[i, 2] for i in [idx_A,idx_B,idx_C]],
})
print("=== Ethiopia recommended Pareto strategies ===")
print(selections.round(4).to_string(index=False))

In [ ]:
# 7.5 White-box validation of the three picks
fig, ax = plt.subplots(figsize=(11, 5.5))
baseline = simulate(NO_INT, params=PARAMS_CAL, ic=IC_CAL, t_max=365, importation=True)
ax.plot(baseline["t"], baseline["I_h"], "k--", lw=1.6, alpha=0.6,
        label=f"No intervention (final C = {baseline['C'].iloc[-1]:.0f})")
colors = {"A: Burden-min": "#c0392b", "B: Knee (balanced)": "#27ae60",
          "C: One Health champion": "#2980b9"}
for _, row in selections.iterrows():
    name = row["Strategy"]
    ctrl = {k: row[k] for k in LEVER_NAMES}
    sim  = simulate(ctrl, params=PARAMS_CAL, ic=IC_CAL, t_max=365, importation=True)
    ax.plot(sim["t"], sim["I_h"], lw=2.5, color=colors[name],
            label=f"{name} (final C={sim['C'].iloc[-1]:.0f})")
ax.set_xlabel("Day"); ax.set_ylabel("Active human Mpox infections")
ax.set_title("Ethiopia — white-box validation of Pareto-optimal strategies")
ax.legend(); plt.tight_layout(); plt.show()

---
## 8. Uncertainty Quantification & Robust Ethiopia Recommendation

Wide bands here are the honest expression of the thin-data reality — a policymaker reading these figures should see the uncertainty, not a false-precision point estimate.

In [ ]:
# 8.1 GP confidence on Pareto solutions
gp = gp_emulator
Xs_pareto = x_scaler.transform(pareto_X)
mu_s, sigma_s = gp.predict(Xs_pareto, return_std=True)
mu = y_scaler.inverse_transform(mu_s)
sigma_cases = sigma_s * y_scaler.scale_[0] if y_scaler.scale_.size > 1 else sigma_s * y_scaler.scale_
ci_lo = mu[:,0] - 1.96 * sigma_cases[:,0]; ci_hi = mu[:,0] + 1.96 * sigma_cases[:,0]
order = np.argsort(pareto_F[:,1])

fig, ax = plt.subplots(figsize=(11, 5))
ax.fill_between(pareto_F[order,1]/1e6, ci_lo[order], ci_hi[order],
                color=COUNTRY_COLOR, alpha=0.25, label="95% GP CI")
ax.plot(pareto_F[order,1]/1e6, mu[order,0], color="#0e6655", lw=2, label="GP mean")
ax.scatter(pareto_F[order,1]/1e6, pareto_F[order,0], s=22,
           color="#e67e22", label="NSGA-II fitness")
ax.set_xlabel("Programme cost (M units)")
ax.set_ylabel("Cumulative cases (1 year)")
ax.set_title("Ethiopia — cost-burden Pareto with GP confidence")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# 8.2 Robust recommendation (min mean + 1 SD)
robust_score = mu[:,0] + sigma_cases[:,0]
idx_robust = int(np.argmin(robust_score))
print("=== Ethiopia ROBUST One Health recommendation ===")
for k, v in zip(LEVER_NAMES, pareto_X[idx_robust]):
    print(f"  {k:6s} = {v:.4f}")
print(f"  Predicted cumulative cases: {mu[idx_robust,0]:.0f} ± {1.96*sigma_cases[idx_robust,0]:.0f}")
print(f"  Cost: {pareto_F[idx_robust,1]/1e6:.2f} M units")
print(f"  Domain imbalance: {pareto_F[idx_robust,2]:.3f}")

In [ ]:
# 8.3 Propagate posterior parameter uncertainty
N_DRAWS = 80
draw_idx = np.random.RandomState(SEED).choice(len(samples), N_DRAWS, replace=False)
robust_lever = pareto_X[idx_robust]
robust_ctrl  = dict(zip(LEVER_NAMES, robust_lever))
burdens = []
for i in draw_idx:
    p_draw = dict(PARAMS_CAL)
    p_draw["beta_hh"] = samples[i,0]; p_draw["beta_rh"] = samples[i,1]
    p_draw["ext_amp"] = samples[i,4]; p_draw["t_peak"]  = samples[i,5]
    p_draw["t_width"] = samples[i,6]
    Nh, Nr = samples[i,2], samples[i,3]
    p_draw["Lambda_h"] = Nh * p_draw["mu_h"]; p_draw["Lambda_r"] = Nr * p_draw["mu_r"]
    ic_draw = {"S_h": Nh-8,"V_h":0.0,"E_h":5.0,"I_h":3.0,"Q_h":0.0,"R_h":0.0,
               "S_r": Nr-30,"I_r":30.0,"R_r":0.0,"C":0.0}
    try:
        df = simulate(robust_ctrl, params=p_draw, ic=ic_draw, t_max=365, importation=True)
        burdens.append(float(df["C"].iloc[-1]))
    except Exception:
        continue
burdens = np.array(burdens)
print("=== Ethiopia posterior-propagated burden under robust recommendation ===")
print(f"  Mean:   {burdens.mean():.0f}")
print(f"  Median: {np.median(burdens):.0f}")
print(f"  95% credible: [{np.quantile(burdens,0.025):.0f}, {np.quantile(burdens,0.975):.0f}]")

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(burdens, bins=25, color=COUNTRY_COLOR, alpha=0.75, edgecolor="white")
ax.axvline(np.median(burdens), color="#c0392b", lw=2.5, label="Posterior median")
ax.axvline(np.quantile(burdens,0.025), color="#2c3e50", lw=1.5, ls="--", label="2.5%/97.5%")
ax.axvline(np.quantile(burdens,0.975), color="#2c3e50", lw=1.5, ls="--")
ax.set_xlabel("Cumulative human Mpox cases under robust Ethiopia recommendation")
ax.set_ylabel("Posterior frequency")
ax.set_title("Ethiopia — parameter-uncertainty-propagated burden")
ax.legend(); plt.tight_layout(); plt.show()

---
## 9. Ablation, Robustness & Tier-Hierarchy Analysis

For Ethiopia we use a **≤5M budget cap** matching the pilot-scale funding constraint from MPSur's questionnaire response.

- **9.1–9.2** — single-seed ablation
- **9.3** — 5-seed × 120-gen robustness check (matches DRC/Nigeria analysis)
- **9.4** — multi-budget scan (does the tier hierarchy hold across budgets?)
- **9.5** — publication-ready tier diagram
- **9.6** — explicit caveat on the animal-domain interpretation

In [ ]:
# 9.1 Constrained ablation
def run_constrained(active_domains, n_gen=60, seed=SEED):
    class P(ElementwiseProblem):
        def __init__(self):
            super().__init__(n_var=5, n_obj=2, n_constr=0,
                             xl=LEVER_BOUNDS[:,0], xu=LEVER_BOUNDS[:,1])
        def _evaluate(self, x, out, *a, **kw):
            x_eff = np.array([x[i] if DOMAIN[LEVER_NAMES[i]] in active_domains else 0.0
                              for i in range(5)])
            xs = x_scaler.transform(x_eff.reshape(1,-1))
            ys = best_emulator.predict(xs)
            if ys.ndim == 1: ys = ys.reshape(1,-1)
            y = y_scaler.inverse_transform(ys)[0]
            out["F"] = [max(y[0],0.0), total_cost(x_eff)]
    res = pymoo_min(P(), NSGA2(pop_size=80, sampling=LHS()),
                    ("n_gen", n_gen), seed=seed, verbose=False)
    return res.F, res.X

scenarios = {
    "Human-only":              {"human"},
    "Animal-only":             {"animal"},
    "Environment-only":        {"environment"},
    "Human+Animal":            {"human","animal"},
    "Human+Environment":       {"human","environment"},
    "Full One Health (all 3)": {"human","animal","environment"},
}
ablation = {}
for name, dom in scenarios.items():
    F, X = run_constrained(dom)
    ablation[name] = {"F": F, "X": X}
    print(f"  {name:30s} -> {len(F)} Pareto solutions")

In [ ]:
# 9.2 Compare Pareto fronts
fig, ax = plt.subplots(figsize=(11, 6))
palette = sns.color_palette("Set1", n_colors=len(scenarios))
for (name, data), color in zip(ablation.items(), palette):
    F = data["F"]; order = np.argsort(F[:,1])
    ax.plot(F[order,1]/1e6, F[order,0], "o-", color=color, alpha=0.85,
            label=name, markersize=5, linewidth=2)
ax.set_xlabel("Programme cost (M units)")
ax.set_ylabel("Cumulative human cases (1 year)")
ax.set_title("Ethiopia — Pareto fronts under different One Health integration levels")
ax.legend(loc="upper right", fontsize=10)
plt.tight_layout(); plt.show()

budget_M = 5.0    # Ethiopia's pilot budget cap
summary = []
for name, data in ablation.items():
    F = data["F"]; feas = F[F[:,1]/1e6 <= budget_M]
    summary.append({"Scenario": name,
                    f"Best cases at <= {budget_M}M": (feas[:,0].min() if len(feas) else float("nan"))})
print(f"\n=== Ethiopia — single-seed ablation at ≤{budget_M}M ===")
print(pd.DataFrame(summary).sort_values(f"Best cases at <= {budget_M}M").to_string(index=False))

In [ ]:
# 9.3 Robustness — 5-seed × 120-gen check
from collections import defaultdict
robust_results = defaultdict(list)
budget_M = 5.0
print(f"Running 5-seed × 120-gen robustness check at ≤{budget_M}M …")
for seed_i in [42, 43, 44, 45, 46]:
    for name, dom in scenarios.items():
        F, _ = run_constrained(dom, n_gen=120, seed=seed_i)
        feas = F[F[:,1]/1e6 <= budget_M]
        robust_results[name].append(feas[:,0].min() if len(feas) else float("nan"))
    print(f"  seed {seed_i} done")

robust_df = pd.DataFrame({"mean": {k: np.mean(v) for k,v in robust_results.items()},
                          "std":  {k: np.std(v)  for k,v in robust_results.items()}}).sort_values("mean")
print(f"\n=== Ethiopia robustness: mean ± std across 5 seeds × 120 gens, ≤{budget_M}M ===")
print(robust_df.round(2))

In [ ]:
# 9.4 Multi-budget scan — does the tier hierarchy hold across budgets?
budgets = [1, 2, 3, 5, 7.5, 10]
multi_budget = {}
for b in budgets:
    row = {}
    for name in scenarios:
        vals = []
        # reuse the last-seed results for each budget by looking at all Pareto points
        for seed_i in [42]:
            F, _ = run_constrained(scenarios[name], n_gen=120, seed=seed_i)
            feas = F[F[:,1]/1e6 <= b]
            vals.append(feas[:,0].min() if len(feas) else float("nan"))
        row[name] = np.mean(vals)
    multi_budget[b] = row
mb_df = pd.DataFrame(multi_budget).T
mb_df.index.name = "Budget_M"
print(f"=== Ethiopia — best-case burden at multiple budget caps ===")
print(mb_df.round(1).to_string())

fig, ax = plt.subplots(figsize=(11, 6))
for name in scenarios:
    ax.plot(mb_df.index, mb_df[name], "o-", lw=2, markersize=8, label=name)
ax.set_xlabel("Programme budget cap (M units)")
ax.set_ylabel("Best achievable cumulative cases")
ax.set_yscale("log")
ax.set_title("Ethiopia — burden vs budget across One Health integration levels")
ax.legend(loc="upper right", fontsize=10)
plt.tight_layout(); plt.show()

### Note on the "Human+Environment = 0.00" emulator-floor artifact

Under the literature-informed Ethiopia calibration ($R_0 \approx 2.5$ within-host but with strong environmental / zoonotic coupling), the response surface at high $\eta_E$ and high $\alpha$ contains almost no positive-burden signal — the emulator's training set is sparse in this regime. NSGA-II therefore hits the `max(y[0], 0.0)` clip in the fitness function and reports **0.00 cumulative cases** for the Human+Environment tier.

The **§7.5 white-box validation** re-simulates the recommended packages using the actual ODE (not the emulator). Under the "One Health champion" strategy the white-box gives **206 cumulative cases** (from a 143 036-case no-intervention baseline — a **694× reduction**). Similarly the burden-minimiser gives 331 cases. **These are the trustworthy Ethiopia burden numbers for the manuscript**, not the emulator's floor at 0.

The **ordering** across tiers in the diagram below (Human+Environment ≤ Full One Health ≪ Human-only plateau ≪ single-domain collapse) is robust — it is driven by mechanistic ODE structure and cost weights, not by emulator noise. The §9.4 multi-budget scan confirms this ordering holds across every budget tested (1M–10M).


In [ ]:
# 9.5 Publication-ready tier hierarchy diagram
means = {n: np.mean(v) for n, v in robust_results.items()}
stds  = {n: np.std(v)  for n, v in robust_results.items()}
order = sorted(means, key=means.get)
m = np.array([means[n] for n in order])
s = np.array([stds[n]  for n in order])

TIER = {
    "Full One Health (all 3)": "#1b5e20",
    "Human+Environment":       "#2e7d32",
    "Human-only":              "#f57c00",
    "Human+Animal":            "#fb8c00",
    "Environment-only":        "#c62828",
    "Animal-only":             "#b71c1c",
}
colors = [TIER.get(n, "#7f8c8d") for n in order]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6),
                                gridspec_kw={"width_ratios":[3,2]})
ypos = np.arange(len(order))
ax1.barh(ypos, m, xerr=s, color=colors, edgecolor="black", capsize=4,
         error_kw=dict(ecolor="#2c3e50", lw=1.5))
ax1.set_yticks(ypos); ax1.set_yticklabels(order)
ax1.set_xscale("log")
ax1.set_xlabel(f"Cumulative cases at ≤{budget_M}M budget (mean ± 1 SD, 5 seeds × 120 gens)")
ax1.set_title("(a) Ethiopia — One Health ablation")
ax1.invert_yaxis()
for i, (mi, si) in enumerate(zip(m, s)):
    ax1.text(mi * 1.05, i, f"{mi:.1f} ± {si:.2f}", va="center", fontsize=10)

sub_order = [n for n in order if means[n] < 1e5]
sub_m = np.array([means[n] for n in sub_order])
sub_s = np.array([stds[n]  for n in sub_order])
sub_c = [TIER.get(n, "#7f8c8d") for n in sub_order]
ypos2 = np.arange(len(sub_order))
ax2.barh(ypos2, sub_m, xerr=sub_s, color=sub_c, edgecolor="black", capsize=4,
         error_kw=dict(ecolor="#2c3e50", lw=1.5))
ax2.set_yticks(ypos2); ax2.set_yticklabels(sub_order)
ax2.set_xlabel("Cases (linear zoom)")
ax2.set_title("(b) Non-catastrophic scenarios — zoom")
ax2.invert_yaxis()
for i, (mi, si) in enumerate(zip(sub_m, sub_s)):
    ax2.text(mi + 0.5, i, f"{mi:.1f} ± {si:.2f}", va="center", fontsize=10)

from matplotlib.patches import Patch
legend_handles = [
    Patch(facecolor="#1b5e20", label="Achievable floor"),
    Patch(facecolor="#f57c00", label="Human-domain plateau"),
    Patch(facecolor="#b71c1c", label="Collapse (no human domain)"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=3,
           bbox_to_anchor=(0.5, -0.02), frameon=False, fontsize=11)
plt.suptitle("Ethiopia One Health ablation — 5-seed robustness analysis",
             fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

### 9.6 Explicit caveat on the animal-domain interpretation

MPSur reports **no animal-domain data collection** in the AI4PEP questionnaire (item 4.4). The animal lever $\eta_A$ is retained in this notebook's ODE for architectural consistency with the Nigeria and DRC pipelines, and the ablation therefore *reports* the animal-domain scenarios above. But:

- **Ethiopia-specific policy recommendations that require animal-domain investment are model-driven only**, not observation-supported.
- If the tier hierarchy above ranks animal-only as catastrophic (which is the model's honest prediction under the calibrated parameters), that reflects the ODE's structural claim that human interventions are necessary — *not* an empirical Ethiopian animal-surveillance finding.
- A future re-run with MPSur's internal case data and any planned animal-domain expansion of their monitoring stack would replace this caveat with a data-supported analysis.

---
## 10. Prospective Policy Dashboard — Ethiopia (MPSur pilot)

One-page summary the pipeline recommends for Ethiopia. Framed as a **prospective deliverable** to the MPSur pilot, not for immediate MoH deployment.

In [ ]:
# 10.1 4-panel dashboard
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) Sobol' driver ranking
levers_sorted = sens_df.sort_values("ST_CumCases", ascending=True)
bar_colors = ["#e67e22" if DOMAIN[L]=="human" else "#27ae60" if DOMAIN[L]=="environment"
              else "#8e44ad" for L in levers_sorted["Lever"]]
axes[0,0].barh(levers_sorted["Lever"], levers_sorted["ST_CumCases"], color=bar_colors)
axes[0,0].set_xlabel("Sobol' total-effect on cumulative cases")
axes[0,0].set_title("(a) Ethiopia Mpox — single-lever driver ranking")

# (b) Pareto + recommended points
ax = axes[0,1]
ax.scatter(pareto_F[:,1]/1e6, pareto_F[:,0], alpha=0.4, c="#bdc3c7", s=22, label="Pareto")
ax.scatter(pareto_F[idx_B,1]/1e6, pareto_F[idx_B,0], s=160, c="#27ae60",
           edgecolor="black", label="Knee (balanced)")
ax.scatter(pareto_F[idx_robust,1]/1e6, pareto_F[idx_robust,0], s=160, c=COUNTRY_COLOR,
           edgecolor="black", label="Robust (UQ-aware)")
ax.set_xlabel("Cost (M units)"); ax.set_ylabel("Cumulative cases")
ax.set_title("(b) Cost-burden Pareto + recommended points")
ax.legend()

# (c) Tier-hierarchy summary from robustness check
tier_order = sorted(means, key=means.get)
tier_colors = [TIER.get(n, "#7f8c8d") for n in tier_order]
axes[1,0].barh(tier_order, [means[n] for n in tier_order],
               xerr=[stds[n] for n in tier_order],
               color=tier_colors, edgecolor="black", capsize=3)
axes[1,0].set_xscale("log")
axes[1,0].set_xlabel(f"Cases at ≤{budget_M}M (5-seed mean ± SD)")
axes[1,0].set_title("(c) One Health integration tiers")
axes[1,0].invert_yaxis()

# (d) Recommended robust mix
levers = LEVER_NAMES
values = pareto_X[idx_robust]
norm = (values - LEVER_BOUNDS[:,0]) / (LEVER_BOUNDS[:,1] - LEVER_BOUNDS[:,0])
colors_d = ["#e67e22" if DOMAIN[L]=="human" else "#27ae60" if DOMAIN[L]=="environment"
            else "#8e44ad" for L in levers]
axes[1,1].bar(levers, norm, color=colors_d, edgecolor="black")
axes[1,1].set_ylabel("Lever effort (normalised 0-1)")
axes[1,1].set_title("(d) Recommended robust Ethiopia One Health package")
axes[1,1].set_ylim(0, 1)
axes[1,1].axhline(0.5, color="grey", ls=":", alpha=0.5)

plt.suptitle("MPSur / Ethiopia — ML-driven One Health Decision Support (prospective)",
             fontsize=15, y=1.02)
plt.tight_layout(); plt.show()

### Citations

- Yinka-Ogunleye A, et al. (2019). *Outbreak of human monkeypox in Nigeria 2017-18*. **Lancet Infectious Diseases**.
- Kibungu EM, et al. (2024). *Clade I-Associated Mpox Cases Associated with Sexual Contact, DRC*. **Emerging Infectious Diseases**.
- WHO AFRO. *Mpox in the African Region — Situation reports 2024–2026*.
- Zhao Z, Wang L, Bergquist R, et al. (2025). *Crafting an innovative one health-aligned machine learning framework for neglected tropical diseases elimination*. **Science in One Health**.
- Deb K, Pratap A, Agarwal S, Meyarivan T (2002). *NSGA-II*. **IEEE Trans. Evol. Comput.**
- Foreman-Mackey D, Hogg DW, Lang D, Goodman J (2013). *emcee: The MCMC Hammer*. **PASP** 125, 306–312.
- Rasmussen CE, Williams CKI (2006). *Gaussian Processes for Machine Learning*. MIT Press.
- Our World in Data — *Mpox (monkeypox)*: https://ourworldindata.org/mpox
- Open-Meteo Historical Weather API: https://open-meteo.com/en/docs/historical-weather-api
- MPSur (Ethiopia partner team) — AI4PEP stakeholder questionnaire response (2026-06-23).
